In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# 1. Set project paths

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = PROJECT_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "performance": REPORTS_DIR / "performance_summary.csv",
    "risk": REPORTS_DIR / "risk_summary.csv",
    "moving": REPORTS_DIR / "moving_average_summary.csv",
    "fundamentals": REPORTS_DIR / "fundamental_summary.csv",
}


In [4]:
# 2. Check that all input files exist

for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name} report: {path}\n"
            "Complete the earlier project step first."
        )

In [6]:
# 3. Load and prepare each report

def load_report(path, prefix):
    data = pd.read_csv(path)

    # Standardize column names.
    data.columns = (
        data.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_")
    )

    if "Ticker" not in data.columns:
        raise ValueError(
            f"'Ticker' column not found in {path.name}. "
            f"Available columns: {list(data.columns)}"
        )

    data["Ticker"] = (
        data["Ticker"].astype(str).str.strip().str.upper()
    )

    data = data.dropna(subset=["Ticker"])

    if data["Ticker"].duplicated().any():
        raise ValueError(
            f"Duplicate tickers found in {path.name}. "
            "Each summary should have one row per stock."
        )

    # Prefix metric names to avoid accidental column collisions.
    data = data.rename(
        columns={
            column: f"{prefix}_{column}"
            for column in data.columns
            if column != "Ticker"
        }
    )

    return data


performance = load_report(FILES["performance"], "performance")
risk = load_report(FILES["risk"], "risk")
moving = load_report(FILES["moving"], "moving")
fundamentals = load_report(FILES["fundamentals"], "fundamentals")



In [7]:
# 4. Combine the reports

comparison = performance.merge(
    risk, on="Ticker", how="outer", validate="one_to_one"
)

comparison = comparison.merge(
    moving, on="Ticker", how="outer", validate="one_to_one"
)

comparison = comparison.merge(
    fundamentals, on="Ticker", how="outer", validate="one_to_one"
)

# Keep the three project stocks in a predictable order.
ticker_order = {"AAPL": 0, "MSFT": 1, "NVDA": 2}

comparison["_sort"] = comparison["Ticker"].map(ticker_order)
comparison = (
    comparison.sort_values("_sort", na_position="last")
    .drop(columns="_sort")
)


In [8]:
# 5. Add readable comparison indicators

def find_column(data, possible_names):
    """Find a column using possible names, ignoring case."""

    lookup = {
        column.lower().replace("_", "").replace("%", "pct"): column
        for column in data.columns
    }

    for name in possible_names:
        key = name.lower().replace("_", "").replace("%", "pct")

        if key in lookup:
            return lookup[key]

    return None


def add_above_below(data, price_col, average_col, output_col):
    """Compare latest price with a moving average."""

    if price_col in data.columns and average_col in data.columns:
        price = pd.to_numeric(data[price_col], errors="coerce")
        average = pd.to_numeric(data[average_col], errors="coerce")

        data[output_col] = pd.Series(
            pd.NA, index=data.index, dtype="string"
        )
        valid = price.notna() & average.notna()

        data.loc[valid, output_col] = (
            price[valid] > average[valid]
        ).map({True: "Above", False: "Below"})


latest_price_col = find_column(
    comparison,
    ["moving_Latest_Adjusted_Close", "Latest_Adjusted_Close"]
)

ma50_col = find_column(comparison, ["moving_MA50", "MA50"])
ma200_col = find_column(comparison, ["moving_MA200", "MA200"])

if latest_price_col and ma50_col:
    add_above_below(
        comparison, latest_price_col, ma50_col, "Price_vs_MA50"
    )

if latest_price_col and ma200_col:
    add_above_below(
        comparison, latest_price_col, ma200_col, "Price_vs_MA200"
    )


In [9]:
# 6. Save the complete comparison report

output_file = REPORTS_DIR / "complete_stock_comparison.csv"
comparison.to_csv(output_file, index=False)

print("\nComplete comparison saved:")
print(output_file)

print("\nAvailable comparison columns:")
print(comparison.columns.tolist())

print("\nCombined report preview:")
print(comparison.to_string(index=False))



Complete comparison saved:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/complete_stock_comparison.csv

Available comparison columns:
['Ticker', 'performance_Start_Date', 'performance_End_Date', 'performance_Trading_Days', 'performance_Starting_Adjusted_Price', 'performance_Ending_Adjusted_Price', 'performance_Total_Return_Pct', 'performance_Annualized_Return_Pct', 'performance_Initial_Investment_USD', 'performance_Ending_Investment_Value_USD', 'performance_Profit_Loss_USD', 'risk_Observations', 'risk_Mean_Daily_Return_Pct', 'risk_Daily_Volatility_Pct', 'risk_Annualized_Volatility_Pct', 'risk_Annualized_Mean_Return_Pct', 'risk_Peak_Date', 'risk_Trough_Date', 'risk_Peak_Price', 'risk_Trough_Price', 'risk_Maximum_Drawdown_Pct', 'moving_Latest_Date', 'moving_Latest_Adjusted_Close', 'moving_MA20', 'moving_Price_Above_MA20', 'moving_MA50', 'moving_Price_Above_MA50', 'moving_MA200', 'moving_Price_Above_MA200', 'fundamentals_Latest_Income_Period', 'fundamentals_Revenu

In [11]:
# 7. Create a chart comparing total returns

return_col = find_column(
    comparison,
    [
        "performance_Total_Return_Pct",
        "performance_Total_Return_Percent",
        "performance_Total_Return",
        "Total_Return_Pct",
        "Total_Return",
    ],
)

if return_col:
    chart_data = comparison[["Ticker", return_col]].copy()
    chart_data[return_col] = pd.to_numeric(
        chart_data[return_col], errors="coerce"
    )
    chart_data = chart_data.dropna()

    if not chart_data.empty:
        fig, ax = plt.subplots(figsize=(9, 6))

        bars = ax.bar(
            chart_data["Ticker"],
            chart_data[return_col],
        )

        ax.axhline(0, linewidth=1)
        ax.set_title(
            "Stock Total Return Comparison",
            fontsize=15,
            fontweight="bold",
        )
        ax.set_xlabel("Stock")
        ax.set_ylabel(f"Total return ({return_col})")
        ax.grid(axis="y", alpha=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.bar_label(bars, fmt="%.2f", padding=3)

        fig.tight_layout()
        fig.savefig(
            FIGURES_DIR / "total_return_comparison.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig)

        print(
            "\nReturn chart saved to "
            "figures/total_return_comparison.png"
        )
else:
    print(
        "\nNo recognized total-return column was found. "
        "Review the performance columns printed above "
        "and adjust the candidate names in the script."
    )

print("\nComparison process completed.")



Return chart saved to figures/total_return_comparison.png

Comparison process completed.
